# Шаг 3. Задания: меняем поведение «умной камеры»

В шаге 2 камера просто показывала всё, что нашла модель. Теперь научим её делать **что-то полезное**
с результатами распознавания:

| Задание | Что делает камера |
|---------|-------------------|
| **A. Счётчик** | считает людей в кадре |
| **B. Фильтр** | показывает только выбранные классы объектов |
| **C. Тревога** | поднимает тревогу, если в кадре телефон |

В каждом задании уже есть рабочий код и строка с пометкой **`ЗАДАНИЕ`**. Её нужно дописать самостоятельно.

> Как и в шаге 2, ноутбук запускается на своём компьютере с веб-камерой. Видео откроется в отдельном окне,
> выход — клавиша `q` (сначала щёлкните по окну с видео). Для Google Colab есть версия `03_challenges_colab.ipynb`.

## 1. Подготовка

Загружаем библиотеки и модель — всё как в прошлых шагах.

In [ ]:
import cv2
from ultralytics import YOLO

MODEL = "yolov8n.pt"
CAMERA = 0  # если камер несколько, попробуйте 1

model = YOLO(MODEL)

## 2. Общий цикл камеры

Во всех трёх заданиях цикл одинаковый: взять кадр → распознать → нарисовать → показать → проверить клавишу `q`.
Отличается только **одно действие**: что сделать с кадром после распознавания.

Поэтому напишем цикл **один раз** в виде функции `run_camera`. Ей передаётся другая функция — `handle`,
которая и решает, что нарисовать на кадре. В каждом задании мы будем писать только свою `handle`.

Функция `handle(picture, names)` получает:
- `picture` — кадр с уже нарисованными рамками (на нём можно рисовать дальше);
- `names` — список названий найденных объектов, например `["person", "person", "cup"]`.

Параметр `classes` нужен для задания B: если его указать, модель будет искать **только** эти классы.

In [ ]:
def run_camera(handle, classes=None, title="YOLO challenge"):
    cap = cv2.VideoCapture(CAMERA)
    if not cap.isOpened():
        raise RuntimeError("Камера не открылась: проверьте подключение или номер CAMERA")

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            result = model(frame, conf=0.4, classes=classes, verbose=False)[0]
            picture = result.plot()
            names = [result.names[int(c)] for c in result.boxes.cls]

            handle(picture, names)  # здесь работает код конкретного задания

            cv2.imshow(title, picture)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)  # на Windows окно иногда не закрывается без этого

### Как рисовать на кадре

В заданиях пригодятся две функции OpenCV:

```python
cv2.putText(picture, "текст", (x, y), cv2.FONT_HERSHEY_SIMPLEX, размер, цвет, толщина)
cv2.rectangle(picture, (x1, y1), (x2, y2), цвет, толщина)
```

**Цвета в OpenCV задаются в порядке BGR** (синий, зелёный, красный), а не RGB:

| Цвет | Значение |
|------|----------|
| красный | `(0, 0, 255)` |
| зелёный | `(0, 255, 0)` |
| синий | `(255, 0, 0)` |
| жёлтый | `(0, 255, 255)` |

> OpenCV умеет рисовать только латиницу — русские буквы превратятся в вопросики. Поэтому надписи на кадре пишем по-английски.

---
## Задание A. Счётчик людей

Функция считает, сколько раз в списке `names` встречается `"person"`, и пишет число зелёным в углу кадра.

**ЗАДАНИЕ:** если людей больше 3, выведите надпись `"Too many!"` **красным**.

<details>
<summary>Подсказка</summary>

Используйте `if people > 3:` и ещё один `cv2.putText`. Поставьте надпись ниже первой, например в точку `(10, 90)`,
чтобы они не наложились друг на друга.
</details>

In [ ]:
def count_people(picture, names):
    people = names.count("person")
    cv2.putText(picture, f"People: {people}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    # ЗАДАНИЕ (count): если людей больше 3, пишите "Too many!" красным


run_camera(count_people)

---
## Задание B. Фильтр классов

Иногда нужно искать не всё подряд, а только определённые предметы. Например, камера на кухне следит только за посудой.
Для этого модели передают параметр `classes` — **список номеров** нужных классов.

Каждый класс COCO имеет свой номер. Несколько примеров:

| Номер | Класс |
|-------|-------|
| 0 | person (человек) |
| 39 | bottle (бутылка) |
| 41 | cup (кружка) |
| 67 | cell phone (телефон) |

Полный список из 80 классов выводит ячейка ниже.

In [ ]:
from labels_ru import ru

for number, name in model.names.items():
    print(f"{number:>2}  {name:<16} {ru(name)}")

Сейчас фильтр показывает бутылки и кружки, а всё остальное (даже людей) модель «не видит».

**ЗАДАНИЕ:** впишите в `FILTER_CLASSES` свои классы. Например, только то, что лежит у вас на столе.

Рисовать дополнительно ничего не нужно, поэтому `show_only` ничего не делает (`pass`). Вся работа — в параметре `classes`.

In [ ]:
FILTER_CLASSES = [39, 41]  # ЗАДАНИЕ (filter): впишите свои классы


def show_only(picture, names):
    pass  # ничего не рисуем дополнительно — фильтрует сама модель


run_camera(show_only, classes=FILTER_CLASSES)

---
## Задание C. Тревога: телефон в кадре

Если модель видит телефон (`"cell phone"`), кадр обводится толстой красной рамкой и появляется надпись `PHONE!`.

`picture.shape[:2]` — высота и ширина кадра в пикселях. Они нужны, чтобы рамка шла точно по краю.

Проблема: модель иногда ошибается на один-два кадра и путает с телефоном, например, пульт.
Из-за этого тревога «мигает».

**ЗАДАНИЕ:** сделайте так, чтобы тревога срабатывала, только если телефон виден **30 кадров подряд** (около секунды).

<details>
<summary>Подсказка</summary>

Нужен счётчик, который «помнит» значение между кадрами. Поэтому он объявлен **снаружи** функции (`phone_frames = 0`),
а внутри функции написано `global phone_frames` — так Python понимает, что мы меняем именно этот счётчик.

- Телефон есть в кадре → увеличиваем счётчик на 1.
- Телефона нет → сбрасываем счётчик в 0.
- Рисуем тревогу, только если `phone_frames >= 30`.
</details>

<details>
<summary>Спойлер: готовое решение (откройте, только если совсем не получается!)</summary>

```python
phone_frames = 0  # сколько кадров подряд виден телефон


def phone_alarm(picture, names):
    global phone_frames

    if "cell phone" in names:
        phone_frames += 1   # телефон есть — счётчик растёт
    else:
        phone_frames = 0    # телефона нет — начинаем считать заново

    if phone_frames >= 30:  # тревога, только если телефон виден ~1 секунду
        h, w = picture.shape[:2]
        cv2.rectangle(picture, (0, 0), (w - 1, h - 1), (0, 0, 255), 20)
        cv2.putText(picture, "PHONE!", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
```

Почему это работает: если модель ошиблась на 1–2 кадра, счётчик не успеет дойти до 30 и сбросится в 0.
А настоящий телефон виден много кадров подряд, и тревога сработает.

Попробуйте вывести на кадр значение счётчика, например `cv2.putText(picture, f"{phone_frames}", ...)`,
и посмотрите, как оно растёт и сбрасывается.
</details>

In [ ]:
phone_frames = 0  # пригодится для задания: сколько кадров подряд виден телефон


def phone_alarm(picture, names):
    global phone_frames

    if "cell phone" in names:
        h, w = picture.shape[:2]
        cv2.rectangle(picture, (0, 0), (w - 1, h - 1), (0, 0, 255), 20)
        cv2.putText(picture, "PHONE!", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
    # ЗАДАНИЕ (alarm): срабатывайте, только если телефон виден 30 кадров подряд


run_camera(phone_alarm)

---
## Дополнительно: придумайте своё

У вас есть готовый `run_camera`. Чтобы получить новую «умную камеру», достаточно написать новую функцию `handle`. Идеи:

1. **Порядок на столе:** надпись `"Clean!"` зелёным, если в кадре нет ни одной кружки и бутылки.
2. **Кот или собака:** выводите `"Cat"` или `"Dog"` в зависимости от того, кто в кадре.
3. **Счётчик всех предметов:** выводите общее число объектов в кадре: `len(names)`.